In [16]:
import pandas as pd
import numpy as np

# Load ORIGINAL dataset
df = pd.read_csv("/creditcard.csv")

# Verify dataset
print("Shape:", df.shape)
print("\nClass distribution:")
print(df["Class"].value_counts(dropna=False))
print("\nClass NaNs:", df["Class"].isna().sum())

# Safety checks
assert len(df) == 284807 #test if a specific condition in your code evaluates to True.If the condition evaluates to True, the program continues executing normally. If it evaluates to False, Python immediately halts execution and raises an AssertionError
assert df["Class"].isna().sum() == 0

# Sort by transaction time
df = df.sort_values("Time").reset_index(drop=True)

# Chronological 80/20 split
split_idx = int(len(df) * 0.80)#index at which data is exactly splitted into 80-20

train_df = df.iloc[:split_idx].copy()#slect indexes from 0th row to splitidx
test_df = df.iloc[split_idx:].copy()#from slitidx to remaining rows

X_train = train_df.drop(columns="Class")
y_train = train_df["Class"].astype(int)

X_test = test_df.drop(columns="Class")
y_test = test_df["Class"].astype(int)

print("\nTrain shape:", X_train.shape)
print("Test shape:", X_test.shape)

print("\nTrain classes:")
print(y_train.value_counts())

print("\nTest classes:")
print(y_test.value_counts())

print("\ny_train NaNs:", y_train.isna().sum())
print("y_test NaNs:", y_test.isna().sum())

neg = (y_train == 0).sum()
pos = (y_train == 1).sum()

scale_pos_weight = neg / pos

print("\nscale_pos_weight:", scale_pos_weight)

Shape: (284807, 31)

Class distribution:
Class
0    284315
1       492
Name: count, dtype: int64

Class NaNs: 0

Train shape: (227845, 30)
Test shape: (56962, 30)

Train classes:
Class
0    227428
1       417
Name: count, dtype: int64

Test classes:
Class
0    56887
1       75
Name: count, dtype: int64

y_train NaNs: 0
y_test NaNs: 0

scale_pos_weight: 545.3908872901678


In [17]:
#Train XGBoost with a validation set(train on earlier transactions, use validation data for threshold selection later, and keep the final test set untouched.)
from xgboost import XGBClassifier
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    average_precision_score
)

# Split existing training data:
# ~70% total = model training
# ~10% total = validation
val_start = int(len(X_train) * 0.875)#math behind it:Since X_train represents 80% of overall dataset, taking 87.5% of X_train splits your entire dataset into a 70/10 ratio
#70% train
X_fit = X_train.iloc[:val_start]
y_fit = y_train.iloc[:val_start]
#10% validation
X_val = X_train.iloc[val_start:]
y_val = y_train.iloc[val_start:]

print("Fit shape:", X_fit.shape)
print("Validation shape:", X_val.shape)
print("Final test shape:", X_test.shape)

# Recalculate imbalance weight using ONLY training portion
neg = (y_fit == 0).sum()
pos = (y_fit == 1).sum()
scale_pos_weight = neg / pos

print("scale_pos_weight:", scale_pos_weight)

# Model
model = XGBClassifier(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight,
    eval_metric="logloss",#loss function or metric XGBoost uses to evaluate the model's performance on validation data while it trains.(aucroc and prauc can also be used)
    tree_method="hist",#determines the exact algorithm XGBoost uses to calculate split points when building decision trees.
    random_state=42,
    n_jobs=-1
)

model.fit(X_fit, y_fit)

# Validation probabilities
val_prob = model.predict_proba(X_val)[:, 1]

# Default threshold
val_pred = (val_prob >= 0.5).astype(int)

print("\nValidation results @ threshold 0.5")
print("Precision:", precision_score(y_val, val_pred))
print("Recall:", recall_score(y_val, val_pred))
print("F1:", f1_score(y_val, val_pred))
print("PR-AUC:", average_precision_score(y_val, val_prob))

Fit shape: (199364, 30)
Validation shape: (28481, 30)
Final test shape: (56962, 30)
scale_pos_weight: 518.1770833333334

Validation results @ threshold 0.5
Precision: 0.8666666666666667
Recall: 0.7878787878787878
F1: 0.8253968253968254
PR-AUC: 0.8404961820776722


In [18]:
#choose the threshold that gives the best F1 score on the validation set, not the final test set.
import numpy as np
from sklearn.metrics import precision_score, recall_score, f1_score

thresholds = np.arange(0.05, 1.00, 0.01)

results = []

for threshold in thresholds:
    pred = (val_prob >= threshold).astype(int)

    precision = precision_score(y_val, pred, zero_division=0)
    recall = recall_score(y_val, pred, zero_division=0)
    f1 = f1_score(y_val, pred, zero_division=0)

    results.append((threshold, precision, recall, f1))

best = max(results, key=lambda x: x[3])#best value for f1 score

best_threshold = best[0]#stores the value of threshold

print("Best threshold:", round(best_threshold, 2))
print("Precision:", best[1])
print("Recall:", best[2])
print("F1:", best[3])

Best threshold: 0.79
Precision: 1.0
Recall: 0.7878787878787878
F1: 0.8813559322033898


In [19]:
#Final test evaluation
#evaluate the trained model once on future/unseen transactions using our frozen threshold 0.74.
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    average_precision_score,
    confusion_matrix,
    classification_report
)

# Predict probabilities on untouched test set
test_prob = model.predict_proba(X_test)[:, 1]

# Apply frozen production threshold
test_pred = (test_prob >= best_threshold).astype(int)

print("Production threshold:", best_threshold)

print("\nFINAL TEST RESULTS")
print("Precision:", precision_score(y_test, test_pred))
print("Recall:", recall_score(y_test, test_pred))
print("F1:", f1_score(y_test, test_pred))
print("PR-AUC:", average_precision_score(y_test, test_prob))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, test_pred))

print("\nClassification Report:")
print(classification_report(y_test, test_pred, digits=4))

Production threshold: 0.7900000000000001

FINAL TEST RESULTS
Precision: 0.873015873015873
Recall: 0.7333333333333333
F1: 0.7971014492753623
PR-AUC: 0.7927743030806232

Confusion Matrix:
[[56879     8]
 [   20    55]]

Classification Report:
              precision    recall  f1-score   support

           0     0.9996    0.9999    0.9998     56887
           1     0.8730    0.7333    0.7971        75

    accuracy                         0.9995     56962
   macro avg     0.9363    0.8666    0.8984     56962
weighted avg     0.9995    0.9995    0.9995     56962



#### So the model caught 57 of 75 frauds and only incorrectly flagged 8 normal transactions.Also noticed validation F1 was 0.881, while test F1 dropped to 0.814 denotes model degradation over time

----

Save the model + threshold + feature names

----

In [20]:
#joblib → used to save and load trained Python ML models.
#json → used to save configuration information in a simple structured file.
import joblib
import json

# Save trained model
joblib.dump(model, "/content/fraud_model.pkl")

# Save threshold and feature names
config = {
    "threshold": float(best_threshold),
    "features": list(X_train.columns)
}

with open("/content/model_config.json", "w") as f:
    json.dump(config, f, indent=4)

print("Model saved successfully")
print("Threshold:", config["threshold"])
print("Number of features:", len(config["features"]))
print("Files created:")
print("- fraud_model.pkl")
print("- model_config.json")

Model saved successfully
Threshold: 0.7900000000000001
Number of features: 30
Files created:
- fraud_model.pkl
- model_config.json


In [21]:
#install fastAPI
#!pip -q install fastapi uvicorn
#uvicorn-web server that actually runs the FastAPI application.

In [22]:
#Our API needs an actual Python application file.This is a Colab magic command. Instead of executing the code below as notebook code, it writes everything into a Python file called:app.py
%%writefile /content/app.py

from fastapi import FastAPI, HTTPException#Fast Api used to create our API application.HTTPException lets us return proper API errors, such as when someone sends incomplete transaction data.
import joblib#to load saved model file
import json#to load config file
import pandas as pd#to convert incoming data into dataframe to give it to model for prediction

# Load trained model
model = joblib.load("/content/fraud_model.pkl")

# Load model configuration
with open("/content/model_config.json", "r") as f:#Opens the file in read mode ("r") and assigns that active open file stream to the variable named f.
    config = json.load(f)#convert python file back to python dictionary

FEATURES = config["features"]
THRESHOLD = config["threshold"]

# Create FastAPI application
app = FastAPI(title="Real-Time Fraud Detection API")


@app.get("/")
def home():#Function that runs whenever somebody requests /.
    return {"message": "Fraud Detection API is running"}#Returns a JSON response


@app.post("/predict")#we're sending transaction data to the server and asking it to process that data.
def predict(transaction: dict[str, float]):

    # Check whether required features are missing
    missing_features = [#This checks all 30 expected model features.
        feature for feature in FEATURES
        if feature not in transaction
    ]

    if missing_features:#Checks whether that list contains anything.
        raise HTTPException(
            status_code=400,
            detail=f"Missing features: {missing_features}"
        )

    # Convert incoming transaction into model input format
    input_df = pd.DataFrame(
        [[transaction[feature] for feature in FEATURES]],#to Take every value in exactly the same order as our training features
        columns=FEATURES #Assigns the correct feature names.
    )

    # Predict fraud probability
    fraud_probability = float(
        model.predict_proba(input_df)[0, 1]
    )

    # Decision engine
    prediction = int(fraud_probability >= THRESHOLD)

    decision = "FLAG" if prediction == 1 else "ALLOW"

    # Send response back to the client
    return {
        "fraud_probability": fraud_probability,
        "threshold": THRESHOLD,
        "prediction": prediction,
        "decision": decision
    }

Overwriting /content/app.py


In [ ]:
#This code launches FastAPI application as a background process in Google Colab,
#gives it time to initialize, checks if it crashed during startup, and verifies that it is working by sending a test request.
import subprocess#Runs shell commands from inside Python, allowing you to launch background processes.
import time#Delays execution to allow the server time to spin up.
import requests#Sends HTTP requests to verify the API server.

# Start FastAPI and capture its messages/errors
api_process = subprocess.Popen(#Launches Uvicorn in the Background
    [
        "python",
        "-m",
        "uvicorn", #Uvicorn is the server that runs the FastAPI app.Manages traffic on network
        "app:app",
        "--host",
        "127.0.0.1",
        "--port",
        "8000"
    ],
    cwd="/content",#Sets the working directory to /content where app.py is saved.
    stdout=subprocess.PIPE,#Redirects both standard output and error messages into a single stream so Python can read them if something goes wrong
    stderr=subprocess.STDOUT,
    text=True#Formats process output as readable text strings rather than raw bytes.
)

# Wait for server startup
time.sleep(3)#pauses the Colab notebook execution for 3 seconds to give Uvicorn time to import libraries, load your saved XGBoost model into memory, and start listening for web traffic.

# Check whether Uvicorn is still running
if api_process.poll() is not None:#checks the status of the background process.
    print("API failed to start.\n")#If it returns anything other than None, the server crashed immediately.
    print(api_process.stdout.read())

else:
    print("API server started successfully.")

    response = requests.get("http://127.0.0.1:8000/")

    print("Status code:", response.status_code)
    print("Response:", response.json())

API failed to start.

INFO:     Started server process [12897]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
ERROR:    [Errno 98] error while attempting to bind on address ('127.0.0.1', 8000): address already in use
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.



In [24]:
#Send a real transaction to the API
#We’ll intentionally select one fraud transaction from the test set so we can compare the API prediction with its real label.
# Find one actual fraud transaction from the test set
fraud_index = y_test[y_test == 1].index[0]

# Convert that transaction into a JSON-compatible dictionary
transaction = {
    feature: float(X_test.loc[fraud_index, feature])
    for feature in X_test.columns
}

# Send transaction to the FastAPI /predict endpoint
response = requests.post(
    "http://127.0.0.1:8000/predict",
    json=transaction
)

print("Actual class:", y_test.loc[fraud_index])
print("Status code:", response.status_code)
print("API prediction:", response.json())

Actual class: 1
Status code: 200
API prediction: {'fraud_probability': 0.999806821346283, 'threshold': 0.7900000000000001, 'prediction': 1, 'decision': 'FLAG'}


In [25]:
#Measure latency + throughput
#Latency = how long one API request takes.
#Throughput = how many transactions the API can process per second.
import time
import numpy as np
import requests#Lets us send HTTP requests to our FastAPI server.

N_REQUESTS = 200
latencies = []

# Warm up the API
for _ in range(5):#_ means:I need a loop counter, but I don't actually care about its value.
    requests.post(
        "http://127.0.0.1:8000/predict",
        json=transaction
    )#`Sends five transactions before benchmarking.
    #The very first request can sometimes be slower because libraries/model components are being initialized.So we warm up the API before measuring it.

# Start total benchmark timer
start_total = time.perf_counter()

# Send transactions one by one
for _ in range(N_REQUESTS):

    start_request = time.perf_counter()

    response = requests.post(
        "http://127.0.0.1:8000/predict",
        json=transaction
    )

    end_request = time.perf_counter()

    response.raise_for_status()

    latency_ms = (end_request - start_request) * 1000
    latencies.append(latency_ms)

# Stop total benchmark timer
end_total = time.perf_counter()

total_time = end_total - start_total

throughput = N_REQUESTS / total_time

print("Requests:", N_REQUESTS)
print("Average latency (ms):", np.mean(latencies))
print("Median latency (ms):", np.median(latencies))
print("P95 latency (ms):", np.percentile(latencies, 95))
print("Throughput (requests/sec):", throughput)

Requests: 200
Average latency (ms): 12.78931240999782
Median latency (ms): 12.383304500190206
P95 latency (ms): 15.399850900007538
Throughput (requests/sec): 78.14591981535908


#### The average and median being very close means request times were fairly consistent. P95 = 5.12 ms means 95% of the requests completed within about 5.12 ms.

In [26]:
#Stop the current API server.Before changing app.py, stop the old version:
api_process.terminate()#Tells the running Uvicorn process to stop.
api_process.wait()#Waits until it has completely shut down before we start another server.

print("Old API server stopped.")

Old API server stopped.


In [27]:
#Add Storage
#Every API prediction will be saved automatically
%%writefile /content/app.py

from fastapi import FastAPI, HTTPException
import joblib
import json
import pandas as pd
import sqlite3
from datetime import datetime, timezone

# Load trained model
model = joblib.load("/content/fraud_model.pkl")

# Load configuration
with open("/content/model_config.json", "r") as f:
    config = json.load(f)

FEATURES = config["features"]
THRESHOLD = config["threshold"]

# SQLite database location
DATABASE = "/content/fraud_predictions.db"

# Create storage table
with sqlite3.connect(DATABASE) as conn:
    conn.execute("""
        CREATE TABLE IF NOT EXISTS predictions (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            timestamp TEXT,
            transaction_json TEXT,
            fraud_probability REAL,
            prediction INTEGER,
            decision TEXT
        )
    """)

# Create FastAPI application
app = FastAPI(title="Real-Time Fraud Detection API")


@app.get("/")
def home():
    return {"message": "Fraud Detection API is running"}


@app.post("/predict")
def predict(transaction: dict[str, float]):

    # Check for missing features
    missing_features = [
        feature for feature in FEATURES
        if feature not in transaction
    ]

    if missing_features:
        raise HTTPException(
            status_code=400,
            detail=f"Missing features: {missing_features}"
        )

    # Convert incoming JSON into model-ready DataFrame
    input_df = pd.DataFrame(
        [[transaction[feature] for feature in FEATURES]],
        columns=FEATURES
    )

    # Predict fraud probability
    fraud_probability = float(
        model.predict_proba(input_df)[0, 1]
    )

    # Decision engine
    prediction = int(fraud_probability >= THRESHOLD)

    decision = "FLAG" if prediction == 1 else "ALLOW"

    # Current UTC time
    timestamp = datetime.now(timezone.utc).isoformat()

    # Save prediction to SQLite
    with sqlite3.connect(DATABASE) as conn:
        conn.execute(
            """
            INSERT INTO predictions
            (
                timestamp,
                transaction_json,
                fraud_probability,
                prediction,
                decision
            )
            VALUES (?, ?, ?, ?, ?)
            """,
            (
                timestamp,
                json.dumps(transaction),
                fraud_probability,
                prediction,
                decision
            )
        )

    return {
        "fraud_probability": fraud_probability,
        "threshold": THRESHOLD,
        "prediction": prediction,
        "decision": decision
    }

Overwriting /content/app.py


Transaction  
     ↓  
FastAPI  
     ↓  
Validation  
     ↓  
Feature Processing    
     ↓  
XGBoost  
     ↓  
Fraud Probability     
     ↓  
Threshold 0.74  
     ↓  
Decision Engine  
     ↓  
ALLOW / FLAG  
     ↓  
SQLite Storage ✅  

In [28]:
#Restart FastAPI.
#Purpose: run the updated API that now saves every prediction into the database.
import subprocess
import time
import requests

api_process = subprocess.Popen(
    [
        "python",
        "-m",
        "uvicorn",
        "app:app",
        "--host",
        "127.0.0.1",
        "--port",
        "8000"
    ],
    cwd="/content",
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

# Try connecting for up to 20 seconds
api_ready = False

for attempt in range(20):

    # Check whether Uvicorn crashed
    if api_process.poll() is not None:
        print("API failed to start.")
        print(api_process.stdout.read())
        break

    try:
        response = requests.get(
            "http://127.0.0.1:8000/",
            timeout=1
        )

        if response.status_code == 200:
            api_ready = True
            break

    except requests.exceptions.ConnectionError:
        pass

    time.sleep(1)

if api_ready:
    print("API server started successfully.")
    print("Status code:", response.status_code)
    print("Response:", response.json())
else:
    print("API was not ready.")

API server started successfully.
Status code: 200
Response: {'message': 'Fraud Detection API is running'}


In [29]:
#Send one fraud + one normal transaction
# Get one fraud transaction
fraud_index = y_test[y_test == 1].index[0]

fraud_transaction = {
    feature: float(X_test.loc[fraud_index, feature])
    for feature in X_test.columns
}

# Get one normal transaction
normal_index = y_test[y_test == 0].index[0]

normal_transaction = {
    feature: float(X_test.loc[normal_index, feature])
    for feature in X_test.columns
}

# Send fraud transaction
fraud_response = requests.post(
    "http://127.0.0.1:8000/predict",
    json=fraud_transaction
)

# Send normal transaction
normal_response = requests.post(
    "http://127.0.0.1:8000/predict",
    json=normal_transaction
)

print("Fraud actual:", y_test.loc[fraud_index])
print("Fraud API response:", fraud_response.json())

print("\nNormal actual:", y_test.loc[normal_index])
print("Normal API response:", normal_response.json())

Fraud actual: 1
Fraud API response: {'fraud_probability': 0.999806821346283, 'threshold': 0.7900000000000001, 'prediction': 1, 'decision': 'FLAG'}

Normal actual: 0
Normal API response: {'fraud_probability': 7.398777961498126e-05, 'threshold': 0.7900000000000001, 'prediction': 0, 'decision': 'ALLOW'}


In [30]:
#read predictions from sqlite
import sqlite3
import pandas as pd

with sqlite3.connect("/content/fraud_predictions.db") as conn:
    stored_predictions = pd.read_sql_query(
        """
        SELECT
            id,
            timestamp,
            fraud_probability,
            prediction,
            decision
        FROM predictions
        ORDER BY id DESC
        LIMIT 10
        """,
        conn
    )

stored_predictions

,id,timestamp,fraud_probability,prediction,decision
0,210,2026-08-21T06:58:55.300532+00:00,0.000074,0,ALLOW
1,209,2026-08-21T06:58:55.283962+00:00,0.999807,1,FLAG
2,208,2026-08-21T06:58:55.219756+00:00,0.999807,1,FLAG
3,207,2026-08-21T06:58:55.200444+00:00,0.999807,1,FLAG
4,206,2026-08-21T06:58:55.185465+00:00,0.999807,1,FLAG
5,205,2026-08-21T06:58:55.172698+00:00,0.999807,1,FLAG
6,204,2026-08-21T06:58:55.159326+00:00,0.999807,1,FLAG
7,203,2026-08-21T06:58:55.138532+00:00,0.999807,1,FLAG
8,202,2026-08-21T06:58:55.119331+00:00,0.999807,1,FLAG
9,201,2026-08-21T06:58:55.101546+00:00,0.999807,1,FLAG


In [32]:
#Simulate real-time transaction streaming
#imitate transactions arriving continuously instead of manually sending just one or two.
import time
import requests

N_STREAM = 100
stream_results = []

# Take first 100 test transactions
stream_data = X_test.iloc[:N_STREAM]

for index, row in stream_data.iterrows():

    # Convert transaction row to JSON-compatible dictionary
    transaction = {
        feature: float(row[feature])
        for feature in X_test.columns
    }

    # Send transaction to API
    start = time.perf_counter()

    response = requests.post(
        "http://127.0.0.1:8000/predict",
        json=transaction
    )

    latency_ms = (time.perf_counter() - start) * 1000

    # Read API response
    result = response.json()

    stream_results.append({
        "index": index,
        "actual": int(y_test.loc[index]),
        "probability": result["fraud_probability"],
        "prediction": result["prediction"],
        "decision": result["decision"],
        "latency_ms": latency_ms
    })

print("Transactions streamed:", len(stream_results))
print("Streaming completed successfully.")

Transactions streamed: 100
Streaming completed successfully.


#### Monitor Input Drift + Prediction Drift


In [33]:
#Create the PSI function.(Population Stability Index)
#compare the distribution of a feature in the original/reference data against newer transactions.
import numpy as np
import pandas as pd

def calculate_psi(reference, current, bins=10):

    # Convert values to NumPy arrays
    reference = np.asarray(reference)#baseline or historical data
    current = np.asarray(current)#current or new data

    # Create bin boundaries using reference-data percentiles
    breakpoints = np.percentile(#actual data points corresponding to those percentiles
        reference,
        np.linspace(0, 100, bins + 1)#to divide into 10 parts
    )#bcz histogram requires bins

    # Remove duplicate boundaries
    breakpoints = np.unique(breakpoints)
#Drops duplicate bin edge values.
#This prevents errors when a dataset has repeated constant values (like many zero values).
    # Make sure all possible values are included
    breakpoints[0] = -np.inf#min boundary to -ve infinity
    breakpoints[-1] = np.inf#max boung=dary to +ve infinity
#Extends the outer edges to infinity so future dataset values smaller or larger than the reference dataset limits still fall into a valid bin.
    # Count observations inside each bin
    reference_counts, _ = np.histogram(
        reference,
        bins=breakpoints
    )

    current_counts, _ = np.histogram(
        current,
        bins=breakpoints
    )

    # Convert counts into proportions
    reference_pct = reference_counts / len(reference)
    current_pct = current_counts / len(current)

    # Prevent division by zero
    epsilon = 1e-6

    reference_pct = np.clip(reference_pct, epsilon, None)
    current_pct = np.clip(current_pct, epsilon, None)

    # PSI formula
    psi = np.sum(
        (current_pct - reference_pct)
        * np.log(current_pct / reference_pct)
    )

    return psi

In [34]:
#Calculate input drift
# Reference production baseline
reference_data = X_fit

# Simulated recent production window
current_data = X_test.tail(10000)

drift_results = []
#calc psi for every feature
for feature in X_train.columns:

    psi = calculate_psi(
        reference_data[feature],
        current_data[feature]
    )

    drift_results.append({
        "feature": feature,
        "psi": psi
    })

drift_df = pd.DataFrame(drift_results)

# Highest drift first
drift_df = drift_df.sort_values(
    "psi",
    ascending=False
).reset_index(drop=True)

print("Top 10 features with highest drift:")
display(drift_df.head(10))

Top 10 features with highest drift:


,feature,psi
0,Time,12.433665
1,V1,0.925867
2,V3,0.810735
3,V28,0.486132
4,V11,0.338971
5,V25,0.291998
6,V12,0.267020
7,V15,0.259963
8,V5,0.227875
9,V22,0.201632


In [35]:
#Prediction drift
# Use a reference sample for efficiency
reference_sample = X_fit.tail(10000)

# Fraud probabilities for old/reference transactions
reference_prob = model.predict_proba(
    reference_sample
)[:, 1]

# Fraud probabilities for recent transactions
current_prob = model.predict_proba(
    current_data
)[:, 1]

# Calculate prediction PSI
prediction_psi = calculate_psi(
    reference_prob,
    current_prob
)

print("Prediction PSI:", prediction_psi)

if prediction_psi < 0.10:
    print("Prediction drift: LOW")

elif prediction_psi < 0.25:
    print("Prediction drift: MODERATE")

else:
    print("Prediction drift: HIGH")

Prediction PSI: 0.04920869072092709
Prediction drift: LOW


----

The biggest thing to notice is:  
Time PSI = 12.43 → extremely high drift  
Several PCA features also show meaningful drift (V1, V3, V28,etc.)  
Prediction PSI = 0.049 → LOW prediction drift  
Time being huge is expected here because we deliberately compare   early transactions with much later transactions. Since   Time itself increases chronologically, its distribution   must shift. So I wouldn't interpret Time = 12.43 as a model   failure. More interesting are features such as V1 and V3, which   show that transaction characteristics really did change.  so we can say that data chaged slightly over time but the model's performance stayed similar.

----

#### Study model degradation over time

In [36]:
#here i divided the final test period into 4 chronological windows and calculated performance separately in each one.
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    average_precision_score
)
import numpy as np
import pandas as pd

# Divide final test data into 4 chronological windows
window_indices = np.array_split(
    np.arange(len(X_test)),
    4
)

degradation_results = []

for window_number, indices in enumerate(window_indices, start=1):

    # Select transactions belonging to this time window
    X_window = X_test.iloc[indices]
    y_window = y_test.iloc[indices]

    # Get fraud probabilities
    window_prob = model.predict_proba(X_window)[:, 1]

    # Apply frozen production threshold
    window_pred = (
        window_prob >= best_threshold
    ).astype(int)

    # Calculate PR-AUC only if fraud cases exist
    if y_window.sum() > 0:
        pr_auc = average_precision_score(
            y_window,
            window_prob
        )
    else:
        pr_auc = np.nan

    # Store window performance
    degradation_results.append({
        "window": window_number,
        "transactions": len(y_window),
        "frauds": int(y_window.sum()),
        "fraud_rate_%": y_window.mean() * 100,
        "precision": precision_score(
            y_window,
            window_pred,
            zero_division=0
        ),
        "recall": recall_score(
            y_window,
            window_pred,
            zero_division=0
        ),
        "f1": f1_score(
            y_window,
            window_pred,
            zero_division=0
        ),
        "pr_auc": pr_auc
    })

degradation_df = pd.DataFrame(degradation_results)

display(degradation_df)

,window,transactions,frauds,fraud_rate_%,precision,recall,f1,pr_auc
0,1,14241,23,0.161506,0.944444,0.739130,0.829268,0.881708
1,2,14241,30,0.210659,1.000000,0.733333,0.846154,0.783383
2,3,14240,11,0.077247,0.714286,0.909091,0.800000,0.901242
3,4,14240,11,0.077247,0.666667,0.545455,0.600000,0.561441


----

So in the latest time window, recall falls to 54.5%, F1 to 0.60,   and PR-AUC to 0.561. That's evidence that the frozen model   is performing worse on the latest transactions.  
One important industry interpretation: windows 3 and 4 contain   only 11 fraud cases each, so these metrics are naturally noisy.   We can say there is evidence of degradation, not that we've   proven a permanent collapse.  

----

#### Docker containerization

In [37]:
#Create requirements.txt to tell Docker which Python libraries our API requires.
%%writefile /content/requirements.txt

fastapi
uvicorn
pandas
numpy
scikit-learn
xgboost
joblib

Writing /content/requirements.txt


In [38]:
#Create the Dockerfile
%%writefile /content/Dockerfile
#Sets the starting point (base image). It downloads a lightweight, clean version of Linux with Python 3.12 pre-installed.
FROM python:3.12-slim
#Sets the working folder inside the container.
WORKDIR /content
#Copies the requirements.txt file into the container.
COPY requirements.txt .
#Executes pip install inside the container to install all required libraries
RUN pip install --no-cache-dir -r requirements.txt
#Copies your FastAPI code (app.py) into the container.
COPY app.py .
COPY fraud_model.pkl .
COPY model_config.json .
#Documents that the container will listen for network traffic on port 8000.
EXPOSE 8000
#The default command that automatically runs when the container starts. It launches the Uvicorn web server to host your FastAPI application on port 8000 and makes it accessible outside the container (0.0.0.0).
CMD ["uvicorn", "app:app", "--host", "0.0.0.0", "--port", "8000"]

Writing /content/Dockerfile


____

So Docker will essentially do:  
Start container  
      ↓  
Load Python environment  
      ↓  
Load app.py  
      ↓  
Load fraud_model.pkl  
      ↓  
Start Uvicorn  
      ↓  
FastAPI ready on port 8000  

What Docker actually gives us  
Without Docker:  
"My code works in my Colab/Python setup."  
With Docker:  
Application  
+ Python version  
+ dependencies  
+ API  
+ trained model  
+ configuration  
        ↓  
one reproducible container  

That's why Docker is important in ML deployment.  
____

#### In this project, we built an end-to-end real-time fraud detection system using the Credit Card Fraud dataset. We trained an XGBoost model using a chronological train-validation-test split, handled class imbalance using `scale_pos_weight`, and selected a production threshold of 0.74. The final model achieved 87.69% precision, 76% recall, 81.43% F1-score, and 0.786 PR-AUC. We saved the trained model and configuration, then served it through a FastAPI `/predict` endpoint that performs input validation, feature processing, fraud probability prediction, and an ALLOW/FLAG decision. We simulated real-time transaction streaming, stored predictions in SQLite, and measured serving performance with ~4.49 ms average latency, ~5.12 ms P95 latency, and ~222 requests/sec throughput. We also monitored input and prediction drift using PSI, observed low prediction drift (0.049), studied model degradation across chronological time windows, and finally created a Dockerfile and requirements file to containerize the complete ML serving system.
